# نقل صوتيات تليجرام تلقائيًا إلى Archive.org

هذا النوتبوك يسحب كل الصوتيات من قناة تليجرام العامة ويرفعها تلقائيًا على **Archive.org** (مقسّمة حسب السنة)، ثم يحفظ ملف `synced_audios.json` و `synced_audios.xlsx` على **Google Drive** جاهزين للاستيراد المباشر في موقعك (Laravel).

**المميزات:**
- لا يستهلك باقة الإنترنت الخاصة بك ولا مساحة جهازك.
- يحفظ التقدم بعد كل ملف على Google Drive (لو الجلسة فصلت، أعد تشغيل الخلية 4 فقط وسيكمل من مكان ما وقف بدون تكرار).
- يسمّي الملفات على Archive بأسماء إنجليزية نظيفة (`msg_1234.m4a`) مع الاحتفاظ بالعنوان العربي والمدة والتاريخ في ملف الاستيراد.

In [ ]:
#@title 1. تثبيت المكتبات وتجهيز مجلد العمل
!pip install -q telethon internetarchive openpyxl

from pathlib import Path

USE_GOOGLE_DRIVE = False #@param {type:"boolean"}
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK_DIR = Path('/content/drive/MyDrive/telegram_archive_sync')
else:
    WORK_DIR = Path('/content/telegram_archive_sync')

WORK_DIR.mkdir(parents=True, exist_ok=True)
TMP_DIR = Path('/tmp/tg_audio')
TMP_DIR.mkdir(parents=True, exist_ok=True)

print('✅ تم التثبيت وتجهيز مجلد العمل بنجاح في:', WORK_DIR)

In [ ]:
#@title 2. إدخال مفاتيح Telegram و Archive.org
#@markdown أدخل المفاتيح من my.telegram.org و archive.org/account/s3.php ثم شغّل هذه الخلية:

TG_API_ID = 0 #@param {type:"integer"}
TG_API_HASH = "" #@param {type:"string"}
IA_ACCESS_KEY = "" #@param {type:"string"}
IA_SECRET_KEY = "" #@param {type:"string"}

#@markdown معرّف القناة (جاهز لقناة محمود الأزهري، أو اكتب @username القناة):
CHANNEL = "-1001171346631" #@param {type:"string"}

#@markdown بادئة اسم العنصر على Archive.org (أحرف إنجليزية صغيرة وأرقام وشرطة فقط):
IA_ITEM_PREFIX = "mahmoud-azhary-audios" #@param {type:"string"}

#@markdown الحد الأقصى للملفات في هذه الجلسة (0 = كل الصوتيات):
MAX_FILES = 0 #@param {type:"integer"}

assert TG_API_ID > 0 and TG_API_HASH.strip(), '❌ من فضلك أدخل TG_API_ID و TG_API_HASH'
assert IA_ACCESS_KEY.strip() and IA_SECRET_KEY.strip(), '❌ من فضلك أدخل مفاتيح Archive.org'

print('✅ الإعدادات جاهزة. القناة:', CHANNEL, '| Archive prefix:', IA_ITEM_PREFIX)

In [ ]:
#@title 3. تسجيل الدخول في تليجرام (مرة واحدة فقط)
from telethon import TelegramClient

SESSION_PATH = str(WORK_DIR / 'tg_session')
client = TelegramClient(SESSION_PATH, TG_API_ID, TG_API_HASH.strip())
await client.start()

me = await client.get_me()
print(f'✅ تم تسجيل الدخول باسم: {me.first_name} (@{me.username})')

In [ ]:
#@title 4. سحب الصوتيات ورفعها تلقائيًا على Archive.org
import asyncio
import json
import os
import re
from pathlib import Path
from telethon.errors import FloodWaitError
from telethon.tl.types import InputMessagesFilterMusic, DocumentAttributeAudio, DocumentAttributeFilename
import internetarchive as ia
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment

CHECKPOINT_FILE = WORK_DIR / 'synced_audios.json'
EXCEL_FILE = WORK_DIR / 'synced_audios.xlsx'

# تحميل التقدم السابق إن وُجد
if CHECKPOINT_FILE.exists():
    synced = json.loads(CHECKPOINT_FILE.read_text(encoding='utf-8'))
else:
    synced = {}

print(f'📦 عدد الملفات المرفوعة سابقًا: {len(synced)}')

# حل كيان القناة
await client.get_dialogs()
channel_ref = int(CHANNEL) if re.fullmatch(r'-?\d+', CHANNEL.strip()) else CHANNEL.strip()
entity = await client.get_entity(channel_ref)
channel_title = getattr(entity, 'title', str(channel_ref))
print(f'📡 جاري القراءة من القناة: {channel_title}')

def extract_audio_meta(msg):
    doc = msg.document
    title = None
    performer = None
    filename = None
    duration = 0

    for attr in (doc.attributes or []):
        if isinstance(attr, DocumentAttributeAudio):
            duration = int(attr.duration or 0)
            if attr.title:
                title = attr.title.strip()
            if attr.performer:
                performer = attr.performer.strip()
        elif isinstance(attr, DocumentAttributeFilename):
            filename = attr.file_name.strip()

    ext = Path(filename).suffix.lower() if filename and Path(filename).suffix else ''
    if not ext:
        mime = (doc.mime_type or '').lower()
        ext_map = {
            'audio/mp4': '.m4a', 'audio/x-m4a': '.m4a', 'audio/m4a': '.m4a',
            'audio/mpeg': '.mp3', 'audio/mp3': '.mp3',
            'audio/ogg': '.ogg', 'audio/aac': '.aac', 'audio/wav': '.wav',
        }
        ext = ext_map.get(mime, '.m4a')

    # اختيار أفضل عنوان عربي نظيف
    clean_name = Path(filename).stem if filename else ''
    caption = (msg.message or '').strip()
    display_title = title or clean_name or (caption.splitlines()[0][:200] if caption else f'تسجيل {msg.id}')
    # تنظيف الشرطات السفلية الزائدة
    display_title = re.sub(r'_+', ' ', display_title).strip()

    return {
        'message_id': msg.id,
        'title': display_title,
        'original_filename': filename or f'msg_{msg.id}{ext}',
        'performer': performer,
        'description': caption,
        'duration': duration,
        'published_at': msg.date.strftime('%Y-%m-%d %H:%M:%S'),
        'year': msg.date.year,
        'ext': ext,
    }

def save_checkpoint_and_excel():
    CHECKPOINT_FILE.write_text(
        json.dumps(synced, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )
    # تصدير ملف إكسل محدّث
    wb = Workbook()
    ws = wb.active
    ws.title = 'صوتيات مرفوعة'
    ws.views.sheetView[0].rightToLeft = True
    headers = ['message_id', 'العنوان', 'المدة (ثانية)', 'التاريخ', 'رابط التشغيل المباشر', 'الوصف']
    ws.append(headers)
    fill = PatternFill(start_color='1B4332', end_color='1B4332', fill_type='solid')
    for c in range(1, len(headers) + 1):
        cell = ws.cell(row=1, column=c)
        cell.fill = fill
        cell.font = Font(bold=True, color='FFFFFF')
        cell.alignment = Alignment(horizontal='center')
    rows = sorted(synced.values(), key=lambda x: x['published_at'], reverse=True)
    for r in rows:
        ws.append([r['message_id'], r['title'], r['duration'], r['published_at'], r['external_url'], r.get('description', '')])
    ws.column_dimensions['A'].width = 12
    ws.column_dimensions['B'].width = 48
    ws.column_dimensions['C'].width = 14
    ws.column_dimensions['D'].width = 22
    ws.column_dimensions['E'].width = 65
    ws.column_dimensions['F'].width = 40
    wb.save(EXCEL_FILE)

uploaded_now = 0
skipped = 0
errors = 0

async for msg in client.iter_messages(entity, filter=InputMessagesFilterMusic):
    if not msg.document:
        continue

    key = str(msg.id)
    if key in synced:
        skipped += 1
        continue

    if MAX_FILES > 0 and uploaded_now >= MAX_FILES:
        print(f'⏸️ وصلنا للحد المطلوب في هذه الجلسة ({MAX_FILES} ملف).')
        break

    meta = extract_audio_meta(msg)
    remote_name = f"msg_{msg.id}{meta['ext']}"
    item_id = f"{IA_ITEM_PREFIX.strip().lower()}-{meta['year']}"
    local_file = TMP_DIR / remote_name

    try:
        print(f"⬇️ [{uploaded_now + 1}] تحميل: {meta['title']} ({meta['published_at'][:10]}) ...")
        while True:
            try:
                await client.download_media(msg, file=str(local_file))
                break
            except FloodWaitError as e:
                print(f'⏳ Telegram FloodWait: انتظار {e.seconds + 2} ثانية...')
                await asyncio.sleep(e.seconds + 2)

        print(f"⬆️ رفع على Archive.org -> {item_id}/{remote_name} ...")
        ia.upload(
            item_id,
            files={remote_name: str(local_file)},
            metadata={
                'title': f"{channel_title} - {meta['year']}",
                'mediatype': 'audio',
                'creator': channel_title,
                'language': 'ara',
            },
            access_key=IA_ACCESS_KEY.strip(),
            secret_key=IA_SECRET_KEY.strip(),
            retries=5,
            retries_sleep=10,
            queue_derive=False,
            verbose=False,
        )

        direct_url = f'https://archive.org/download/{item_id}/{remote_name}'
        synced[key] = {
            'message_id': meta['message_id'],
            'title': meta['title'],
            'original_filename': meta['original_filename'],
            'performer': meta['performer'],
            'description': meta['description'],
            'duration': meta['duration'],
            'published_at': meta['published_at'],
            'archive_item': item_id,
            'external_url': direct_url,
        }
        save_checkpoint_and_excel()
        uploaded_now += 1
        print(f'✅ تم ({len(synced)} إجمالي): {direct_url}')

    except Exception as ex:
        errors += 1
        print(f'❌ خطأ في msg_{msg.id} ({meta["title"]}): {ex}')
    finally:
        if local_file.exists():
            local_file.unlink()

save_checkpoint_and_excel()
print(f'\n🎉 انتهت الجلسة! مرفوع الآن: {uploaded_now} | متخطى سابقًا: {skipped} | أخطاء: {errors} | الإجمالي المحفوظ: {len(synced)}')
print('📄 ملف JSON للاستيراد في Laravel:', CHECKPOINT_FILE)
print('📊 ملف Excel:', EXCEL_FILE)